# Notebook 03 - Synthetic Signal Analysis

This notebook analyzes invented values from anonymous signal channels. Results are illustrative and are not sourced from an organization or operating asset.

## Analysis step 2

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import joblib
from sklearn.preprocessing import RobustScaler

ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'interim' / 'channel_values.csv').is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'interim' / 'channel_values.csv').is_file():
    raise FileNotFoundError('data/interim/channel_values.csv not found. Run notebook 01 first.')

INTERIM = ROOT / 'data' / 'interim'
FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.constrained_layout.use': True,
})
ACCENT, NEUTRAL, GREY = '#E84B4B', '#4878D0', '#AAAAAA'

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 200)

## Analysis step 3

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
SIGNAL_THRESH = 5.0
PARTIAL_FRAC = 0.80
GROUP_LOW_FRAC = 0.10
RESTART_DAYS = 3
MAX_FILL_GAP = 3
CLIP_NEGATIVES = False
BASELINE_YEAR = 2032

CORE_CHANNELS = [
    'channel_098', 'channel_103', 'channel_102',
    'channel_105', 'channel_106', 'channel_107',
    'channel_101', 'channel_108', 'channel_100',
    'channel_104', 'channel_095', 'channel_094',
    'channel_093', 'channel_099',
]
FEED_COLS = ['channel_001', 'channel_004', 'channel_002',
             'channel_003', 'channel_070', 'channel_096', 'channel_097']
GROUP_CHANNELS = [f'channel_{i:03d}' for i in range(8, 15)]

## Analysis step 4

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
df = pd.read_csv(INTERIM / 'channel_values.csv', index_col=0, parse_dates=True)
channel = pd.read_csv(INTERIM / 'channel_dictionary.csv')
kd = pd.read_csv(INTERIM / 'channel_selection.csv')

print('values:', df.shape)
print('range:', df.index.min().date(), 'to', df.index.max().date())
print('sorted:', df.index.is_monotonic_increasing)

missing_needed = [c for c in CORE_CHANNELS + FEED_COLS + GROUP_CHANNELS if c not in df.columns]
print('required anonymous channels missing:', missing_needed)
assert not missing_needed

## Analysis step 5

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
def find_blocks(mask):
    """Contiguous True runs of a daily boolean Series -> start, end, days."""
    mask = mask.fillna(False).astype(bool)
    block_id = (mask != mask.shift()).cumsum()
    rows = [{'start': g.index.min(), 'end': g.index.max(), 'days': len(g)}
            for _, g in mask[mask].groupby(block_id[mask])]
    return pd.DataFrame(rows, columns=['start', 'end', 'days'])


top, bott = df['channel_103'], df['channel_102']
print('days with a missing asset signal:', int((top.isna() | bott.isna()).sum()))

signal_low = (top < SIGNAL_THRESH) & (bott < SIGNAL_THRESH)      # NaN -> False

blocks_signal = find_blocks(signal_low)
print(f'signal rule: {int(signal_low.sum())} days in {len(blocks_signal)} blocks')
blocks_signal

In [ ]:
normal_top = top[~signal_low].median()
context_total = df[GROUP_CHANNELS].sum(axis=1, min_count=1)
context_low = context_total < context_total.median() * GROUP_LOW_FRAC

rules = pd.DataFrame({
    'signal_low': signal_low,
    'signal_partial': (top < PARTIAL_FRAC * normal_top) & ~signal_low,
    'context_inactive': context_total <= 0,
    'context_low': context_low,
}).fillna(False).astype(bool)

print(f'normal signal median on active days: {normal_top:.2f}')
print(f'median aggregate context signal: {context_total.median():.2f}')
print()

check = pd.DataFrame({
    'days': rules.sum(),
    'also_signal_low': [int((rules[c] & rules['signal_low']).sum()) for c in rules],
    'not_signal_low': [int((rules[c] & ~rules['signal_low']).sum()) for c in rules],
})
check

In [ ]:
fig, ax = plt.subplots(figsize=(14, 2.8))
for i, col in enumerate(rules.columns):
    d = rules.index[rules[col]]
    ax.vlines(d, i - 0.4, i + 0.4, color=ACCENT if col == 'signal_low' else NEUTRAL, lw=1.2)
ax.set_yticks(range(len(rules.columns)))
ax.set_yticklabels(rules.columns)
ax.set_ylim(-0.7, len(rules.columns) - 0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_title('Which days each shutdown signal flags (red = the signal rule used in the EDA)')
plt.savefig(FIG_DIR / 'notebook_03_plot_01.png', bbox_inches='tight')
plt.show()

## Analysis step 6

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
is_shutdown = signal_low.copy()
is_suspect = rules[['signal_partial', 'context_inactive', 'context_low']].any(axis=1) & ~is_shutdown

blocks_suspect = find_blocks(is_suspect)
print(f'synthetic low-state periods: {int(is_shutdown.sum())} days in {len(blocks_signal)} blocks')
print(f'other flagged periods: {int(is_suspect.sum())} days in {len(blocks_suspect)} blocks')
print(f'combined flagged days: {int((is_shutdown | is_suspect).sum())}')
blocks_suspect

### Analysis step 7

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
alt = (is_shutdown != is_shutdown.shift()).cumsum()
seg_raw = alt.where(~is_shutdown)
segment_id = seg_raw.map({old: i + 1 for i, old in enumerate(seg_raw.dropna().unique())})

day_in_segment = pd.Series(np.nan, index=df.index)
is_restart = pd.Series(False, index=df.index)
seg_rows = []
for sid in segment_id.dropna().unique():
    m = (segment_id == sid).to_numpy()
    pos = np.flatnonzero(m)
    day_in_segment.iloc[pos] = np.arange(len(pos))
    after_shutdown = pos[0] > 0 and bool(is_shutdown.iloc[pos[0] - 1])
    if after_shutdown:
        is_restart.iloc[pos[:RESTART_DAYS]] = True
    seg_rows.append({'segment_id': int(sid), 'start': df.index[pos[0]], 'end': df.index[pos[-1]],
                     'days': len(pos), 'starts_after_shutdown': after_shutdown})

segments = pd.DataFrame(seg_rows)
print(f'{len(segments)} operating segments, {int(is_restart.sum())} restart days flagged')
segments

In [ ]:
events = blocks_signal.copy()
n = len(events)
if n:
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows), squeeze=False)
    for ax, (_, ev) in zip(axes.flatten(), events.iterrows()):
        w = df.loc[ev['start'] - pd.Timedelta(days=10): ev['end'] + pd.Timedelta(days=10)]
        ax.plot(w.index, w['channel_103'], color=NEUTRAL, lw=1.2, label='top signal')
        ax.axvspan(ev['start'], ev['end'], color=ACCENT, alpha=0.25, lw=0)
        ax.axhline(SIGNAL_THRESH, color=GREY, ls='--', lw=0.8)
        ax2 = ax.twinx()
        ax2.plot(w.index, w['channel_098'], color='#2ca02c', lw=1.0, label='signal')
        ax2.spines['right'].set_visible(True)
        ax.set_title(f"{ev['start'].date()} -> {ev['end'].date()} ({ev['days']} d)", fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    for ax in axes.flatten()[n:]:
        ax.set_visible(False)
    fig.suptitle('Each shutdown with 10 days before and after (blue = top signal, green = signal)', fontsize=10)
    plt.savefig(FIG_DIR / 'notebook_03_plot_02.png', bbox_inches='tight')
    plt.show()

### Analysis step 8

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
cols = CORE_CHANNELS + FEED_COLS
tab = pd.DataFrame({
    'pct_missing_operating': df.loc[~is_shutdown, cols].isna().mean() * 100,
    'pct_missing_shutdown':  df.loc[is_shutdown, cols].isna().mean() * 100,
}).round(1)
print(tab[(tab > 0).any(axis=1)])

context_gap = df[FEED_COLS].isna()
print()
print('days with at least one context column missing :', int(context_gap.any(axis=1).sum()))
print('days with all context columns missing         :', int(context_gap.all(axis=1).sum()))
print('days with channel_099 missing                 :', int(df['channel_099'].isna().sum()))
print('of those, also a shutdown day              :', int((df['channel_099'].isna() & is_shutdown).sum()))

## Analysis step 9

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 10

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
drop_cols = kd.loc[kd['eda_decision'].str.startswith('DROP'), 'name'].tolist()

dropped = pd.DataFrame({
    'name': drop_cols,
    'pct_missing': (df[drop_cols].isna().mean() * 100).round(1).values,
    'reason': ['Excluded by the synthetic channel selection'] * len(drop_cols),
})
print(f'{len(drop_cols)} channels selected for exclusion:')
print(dropped[['name', 'pct_missing', 'reason']].to_string(index=False))

In [ ]:
# safety checks against the EDA decisions
kd_keep = kd.loc[kd['eda_decision'].str.startswith(('KEEP', 'FLAG', 'REVIEW')), 'name']
wrongly_dropped = sorted(set(kd_keep) & set(drop_cols))
print('KEEP/FLAG/REVIEW columns that would be dropped:', wrongly_dropped)
assert not wrongly_dropped

kd_drop = kd.loc[kd['eda_decision'].str.startswith('DROP'), 'name']
not_covered = sorted(set(kd_drop) - set(drop_cols))
print('DROP columns of the EDA table not covered by the pattern:', not_covered)
assert not not_covered

assert not set(CORE_CHANNELS + FEED_COLS) & set(drop_cols)

df = df.drop(columns=drop_cols)
sensor_cols = list(df.columns)
print()
print(f'kept {len(sensor_cols)} sensor columns, dropped {len(drop_cols)}')

## Analysis step 11

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 12

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
op_mask = ~is_shutdown
neg = pd.DataFrame({
    'signal_negative':          (df['channel_098'] < 0) & op_mask,
    'signal': (df['channel_104'] < 0) & op_mask,
})

print('negative days on operating days:')
print(neg.sum().to_string())
print()
print('by quarter:')
print(neg.groupby(neg.index.to_period('Q')).sum().to_string())

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, col, flag in [(axes[0], 'channel_098', 'signal_negative'),
                      (axes[1], 'channel_104', 'signal')]:
    s = df[col].where(op_mask)
    ax.plot(s.index, s, color=NEUTRAL, lw=0.8)
    ax.scatter(s.index[neg[flag]], s[neg[flag]], color=ACCENT, s=14, zorder=3, label='negative')
    for _, ev in blocks_signal.iterrows():
        ax.axvspan(ev['start'], ev['end'], color=GREY, alpha=0.3, lw=0)
    ax.axhline(0, color='black', ls='--', lw=0.7)
    ax.set_ylabel(col)
    ax.legend(loc='upper left')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.suptitle('Negative values (red) on operating days, grey = shutdown')
plt.savefig(FIG_DIR / 'notebook_03_plot_03.png', bbox_inches='tight')
plt.show()

## Analysis step 13

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
flags_neg = neg.copy()

if CLIP_NEGATIVES:
    for col in ['channel_098', 'channel_104']:
        df[col] = df[col].clip(lower=0)
    print('negatives clipped at 0')
else:
    print('negatives kept as measured (flags only)')

## Analysis step 14

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
def fill_short_gaps(s, max_gap):
    """Forward-fill NaN runs of length <= max_gap. Longer runs and leading NaN stay NaN."""
    isna = s.isna()
    run_id = (isna != isna.shift()).cumsum()
    run_len = isna.groupby(run_id).transform('sum')
    fillable = isna & (run_len <= max_gap)
    out = s.copy()
    out[fillable] = s.ffill()[fillable]
    return out

# quick test on a toy series: gap of 2 is filled, gap of 5 is not, leading NaN stays
toy = pd.Series([np.nan, 1, np.nan, np.nan, 2, np.nan, np.nan, np.nan, np.nan, np.nan, 3.0])
print(pd.DataFrame({'before': toy, 'after': fill_short_gaps(toy, 3)}).T.to_string())

In [ ]:
clean = df.copy()
imputed = pd.DataFrame(False, index=clean.index, columns=clean.columns)

for sid in segment_id.dropna().unique():
    idx = clean.index[(segment_id == sid).to_numpy()]
    block = clean.loc[idx]
    filled = block.apply(fill_short_gaps, max_gap=MAX_FILL_GAP)
    imputed.loc[idx] = (block.isna() & filled.notna()).to_numpy()
    clean.loc[idx] = filled

n_imputed_total = int(imputed.values.sum())
print('cells filled:', n_imputed_total)
print('cells filled on shutdown days (must be 0):', int(imputed[is_shutdown].values.sum()))
print()
left = clean.loc[~is_shutdown, sensor_cols].isna().sum()
left = left[left > 0].sort_values(ascending=False)
print('columns that still have missing values on operating days:')
print(left.to_string() if len(left) else 'none')

## Analysis step 15

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 16

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
c3 = clean['channel_099']
print('negative channel_099 values:', int((c3 < 0).sum()), '| zeros:', int((c3 == 0).sum()))
clean['channel_099_log'] = np.log1p(c3.clip(lower=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
s = c3[~is_shutdown].dropna()
axes[0].hist(s, bins=60, color=NEUTRAL); axes[0].set_title('channel_099')
axes[1].hist(np.log1p(s.clip(lower=0)), bins=60, color=NEUTRAL); axes[1].set_title('log1p(channel_099)')
plt.savefig(FIG_DIR / 'notebook_03_plot_04.png', bbox_inches='tight')
plt.show()

## Analysis step 17

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
clean['is_shutdown']          = is_shutdown
clean['is_restart']           = is_restart
clean['is_suspect']           = is_suspect
clean['segment_id']           = segment_id.astype('Int64')
clean['day_in_segment']       = day_in_segment.astype('Int64')
clean['signal_negative']         = flags_neg['signal_negative']
clean['signal'] = flags_neg['signal']
clean['n_imputed']            = imputed.sum(axis=1).astype(int)
clean['core_complete']        = (~is_shutdown) & clean[CORE_CHANNELS].notna().all(axis=1)
clean['is_baseline']          = (clean.index.year == BASELINE_YEAR) & ~is_shutdown & ~is_restart

flag_cols = ['is_shutdown', 'is_restart', 'is_suspect', 'segment_id', 'day_in_segment',
             'signal_negative', 'signal', 'n_imputed', 'core_complete', 'is_baseline']

print('operating days      :', int((~clean['is_shutdown']).sum()))
print('core_complete days  :', int(clean['core_complete'].sum()))
print('baseline days       :', int(clean['is_baseline'].sum()))
print('2033 operating days :', int(((clean.index.year == 2033) & ~clean['is_shutdown']).sum()))

## Analysis step 18

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
scale_cols = [c for c in sensor_cols if c != 'channel_099'] + ['channel_099_log']
base = clean.loc[clean['is_baseline'], scale_cols]
print('rows used to fit the scaler:', len(base))

iqr = base.quantile(0.75) - base.quantile(0.25)
flat = iqr[iqr <= 1e-9].index.tolist()
print('columns with IQR = 0 in the baseline (the scaler cannot scale these):', flat)

scaler = RobustScaler().fit(base)
scaled = pd.DataFrame(scaler.transform(clean[scale_cols]), index=clean.index, columns=scale_cols)

print('max |median| of the scaled baseline:', float(scaled.loc[clean['is_baseline']].median().abs().max().round(6)))

In [ ]:
# preview: how far does 2033 sit from the healthy reference, in robust units?
op25 = (clean.index.year == 2033) & ~clean['is_shutdown'].to_numpy()
shift = scaled.loc[op25].median().sort_values(key=lambda s: s.abs(), ascending=False)
print('median of the 2033 operating days in baseline units (top 12):')
print(shift.head(12).round(2).to_string())

## Analysis step 19

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 20

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
final_cols = sensor_cols + ['channel_099_log'] + flag_cols
clean = clean[final_cols]

assert clean.index.is_monotonic_increasing and clean.index.is_unique
assert len(clean) == len(df)
assert not set(drop_cols) & set(clean.columns)
assert not np.isinf(clean[sensor_cols + ['channel_099_log']].to_numpy(dtype=float)).any()
assert clean[['is_shutdown', 'is_restart', 'is_suspect', 'core_complete', 'is_baseline']].notna().all().all()
assert clean.loc[clean['is_shutdown'], 'n_imputed'].eq(0).all()
assert list(scaled.columns) == scale_cols and scaled.index.equals(clean.index)

print('clean  :', clean.shape)
print('scaled :', scaled.shape)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
op = clean[~clean['is_shutdown']]

axes[0].plot(op.index, op['channel_098'], color=NEUTRAL, lw=0.8)
axes[0].plot(op.index, op['channel_098'].rolling(14, min_periods=5).mean(), color=ACCENT, lw=1.4)
axes[0].axhline(0, color='black', ls='--', lw=0.7)
axes[0].set_ylabel('channel_098')

axes[1].plot(op.index, op['channel_099_log'], color=NEUTRAL, lw=0.8)
axes[1].plot(op.index, op['channel_099_log'].rolling(14, min_periods=5).mean(), color=ACCENT, lw=1.4)
axes[1].set_ylabel('log1p(channel_099)')

state = clean[['is_shutdown', 'is_restart', 'is_suspect', 'n_imputed']].copy()
state['imputed'] = state['n_imputed'] > 0
for i, (col, color) in enumerate([('is_shutdown', ACCENT), ('is_restart', '#FFA500'),
                                  ('is_suspect', '#9467bd'), ('imputed', '#2ca02c')]):
    d = state.index[state[col]]
    axes[2].vlines(d, i - 0.4, i + 0.4, color=color, lw=1.2)
axes[2].set_yticks(range(4)); axes[2].set_yticklabels(['shutdown', 'restart', 'suspect', 'imputed'])

for ax in axes[:2]:
    for _, ev in blocks_signal.iterrows():
        ax.axvspan(ev['start'], ev['end'], color=GREY, alpha=0.3, lw=0)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.suptitle('Clean table: signal, log signal and the day flags (grey = shutdown)')
plt.savefig(FIG_DIR / 'notebook_03_plot_05.png', bbox_inches='tight')
plt.show()

## Analysis step 21

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
clean.to_csv(INTERIM / 'channel_clean.csv')
scaled.to_csv(INTERIM / 'channel_scaled.csv')
joblib.dump({'scaler': scaler, 'columns': scale_cols, 'baseline_year': BASELINE_YEAR,
             'n_baseline_rows': int(len(base))}, INTERIM / 'channel_scaler.joblib')

ev_out = blocks_signal.copy()
ev_out['rule'] = f'top and bottom signal < {SIGNAL_THRESH}'
ev_out.to_csv(INTERIM / 'state_segments.csv', index=False)
dropped.to_csv(INTERIM / 'dropped_columns.csv', index=False)

summary = pd.Series({
    'rows': len(clean),
    'sensor_columns_in': len(sensor_cols) + len(drop_cols),
    'sensor_columns_dropped': len(drop_cols),
    'sensor_columns_kept': len(sensor_cols),
    'shutdown_days': int(clean['is_shutdown'].sum()),
    'shutdown_blocks': len(blocks_signal),
    'suspect_days': int(clean['is_suspect'].sum()),
    'operating_segments': len(segments),
    'restart_days': int(clean['is_restart'].sum()),
    'cells_imputed': n_imputed_total,
    'signal_negative_days': int(clean['signal_negative'].sum()),
    'signal': int(clean['signal'].sum()),
    'core_complete_days': int(clean['core_complete'].sum()),
    'baseline_days': int(clean['is_baseline'].sum()),
    'clip_negatives': CLIP_NEGATIVES,
}, name='value')
summary.to_csv(INTERIM / 'cleaning_summary.csv', header=True)

print('saved:', sorted(p.name for p in INTERIM.iterdir()))
summary

## Analysis step 22

This section uses synthetic values and anonymous channel identifiers.